Ch 1-3: Array, Strings, Linked Lists, Stacks, and Queues <small>-- [Open in Colab](https://colab.research.google.com/github/henry4j/-/blob/main/episode_1-3.ipynb)

Python Basics: [data structures](https://docs.python.org/3/tutorial/datastructures.html), [deque](https://docs.python.org/3/library/collections.html#collections.deque), [heapq](https://docs.python.org/3/library/heapq.html), [string methods](https://docs.python.org/3.4/library/stdtypes.html#string-methods), [itertools](https://docs.python.org/3/library/itertools.html#itertools-recipes), [functools](https://docs.python.org/3/library/functools.html), [match-case](https://www.inspiredpython.com/course/pattern-matching/mastering-structural-pattern-matching), [new python features](https://www.nicholashairs.com/posts/major-changes-between-python-versions/), [python tricks](https://sahandsaba.com/14-more-python-features-and-tricks-you-may-not-know.html)

In [ ]:
# @title ##### 1.1 Write a program to determine if a string has all unique characters. What if you cannot use additional data structures?
def uniq(s) -> bool:  # time: O(n), space(n)
  return all(v == 1 for v in histogram(s).values())

def histogram(iterable) -> dict:
  h = {}
  for e in iterable:
    h[e] = h.get(e, 0) + 1
  return h

def uniqq(s) -> bool:  # time: o(n^2), space: o(1)
  for i_lhs, lhs in enumerate(s):
    for i_rhs in range(i_lhs + 1, len(s)):
      if lhs == s[i_rhs]:
        return False
  return True


assert all([uniq(""), uniq("a"), uniq("ab"), not uniq("aa"), not uniq("aba")])
assert all([uniqq(""), uniqq("a"), uniqq("ab")])
assert not any([uniqq("aa"), uniqq("aba")])

In [ ]:
# @title ###### 1.2 Write a program to test if two strings are permutations of each other.
def anagram(s1, s2):
    if len(s1) != len(s2):
        return False
    h = histogram(s1)
    for e in s2:
        if h.get(e, 0) > 0:
            h[e] -= 1
        else:
            return False
    return True

def anagram2(s1, s2):
    signature = lambda s: ''.join(sorted(s))
    return len(s1) == len(s2) and signature(s1) == signature(s2)

def anagram3(s1, s2):
    return histogram(s1) == histogram(s2)

assert anagram('', '') and anagram('a', 'a') and anagram('ab', 'ba')
assert anagram('aab', 'aba') and anagram('aabb', 'abab')
assert not anagram('a', '') and not anagram('', 'a')
assert not anagram('a', 'b') and not anagram('aa', 'ab')
assert anagram2('aab', 'aba') and anagram2('aabb', 'abab')
assert anagram3('aab', 'aba') and anagram3('aabb', 'abab')

In [ ]:
# @title ##### 1.3 Write a program to replace all spaces in a string with %20.
def resize(L, new_size, fill_value=None):
  del L[new_size:]
  L.extend([fill_value] * (new_size - len(L)))
  return L

def escape_spaces(s):
  L, m = list(s), len(s)  # m: input length.
  resize(L, n := len(L) + 2 * L.count(" "), " ")  # n: output length.
  w = n
  for r in range(m - 1, -1, -1):  # i in [m-1, 0].
    buff = "%20" if L[r] == " " else [L[r]]
    w -= len(buff)
    L[w : w + len(buff)] = buff
  return "".join(L)

assert "a%20b%20c%20" == escape_spaces("a b c ")

In [ ]:
# @title ##### 1.4 Write a program to test if a string is a permutation of a palindrome.
def histogram(iterable) -> dict:
  h = {}
  for e in iterable:
    h[e] = h.get(e, 0) + 1
  return h

palindormic = lambda s: sum(v % 2 for v in histogram(s.lower()).values()) < 2
assert all([palindormic(""), palindormic("a"), palindormic("aa"), palindormic("aba")])
assert not any([palindormic("ab"), palindormic("abbc")])

In [ ]:
# @title ##### 1.5 Write a program to test if two strings are one or zero edits away from each other (insert, delete, or replace).
from functools import lru_cache
from enum import Enum
Edit = Enum("Edit", dict(I="Insert", D="Delete", R="Replace", M="Match"))

def recap(kv, k, v) -> tuple[int, tuple]:  # k: cost, v: steps.
  return (kv[0] + k, kv[1] + v)

def edit(a, b):
  @lru_cache(maxsize=None)
  def prog(m, n):
    if m == 0 or n == 0:
      return (m + n, ((Edit.D,) * m if m > 0 else (Edit.I,) * n))
    c = int(a[m - 1] != b[n - 1])  # edit cost: 1 or 0.
    cases = (
      recap(prog(m - 1, n - 1), c, ([Edit.M, Edit.R][c],)),
      recap(prog(m - 1, n - 0), 1, (Edit.D,)),
      recap(prog(m - 0, n - 1), 1, (Edit.I,)),
    )
    return min(cases, key=lambda cost_steps: cost_steps[0])
  return prog(len(a), len(b))

expected = (3, (Edit.R, Edit.M, Edit.M, Edit.M, Edit.R, Edit.M, Edit.I))
assert expected == edit("kitten", "sitting")
assert (3, (Edit.D,) * 3) == edit("cat", "")
assert (3, (Edit.I,) * 3) == edit("", "sit")

In [ ]:
# @title ##### 1.6 Write a method to compress a string using counts of repeated chars, e.g., aabcccccaaa becomes a2b1c5a3.
def compressed(s):
  s2, m, start = [], len(s), 0
  for stop in range(1, m + 1):
    if stop == m or s[start] != s[stop]:
      s2.extend([s[start], str(stop - start)])
      start = stop
  return "".join(s2) if len(s2) < m else s

assert "a2b1c5a3" == compressed("aabcccccaaa")
assert "abcc" == compressed("abcc")
assert "abc" == compressed("abc")
assert "" == compressed("")

In [ ]:
# @title ##### 1.7 Given an image represented by an NxN matrix, write a program to rotate the image by 90 degrees; in-place, in O(1) space.
def rotate(g):
  n = len(g)
  for layer in range(n // 2):
    head, tail = layer, n - 1 - layer
    for i in range(head, tail):
      top = g[layer][i]
      g[layer][i] = g[n - 1 - i][head]  # left to top
      g[n - 1 - i][head] = g[tail][n - 1 - i]  # bottom to left
      g[tail][n - 1 - i] = g[i][tail]  # right to bottom
      g[i][tail] = top  # top to right
  return g

g = [
  [1, 2, 3, 4, 5],
  [6, 7, 8, 9, 10],
  [11, 12, 13, 14, 15],
  [16, 17, 18, 19, 20],
  [21, 22, 23, 24, 25],
]
expected = [
  [21, 16, 11, 6, 1],
  [22, 17, 12, 7, 2],
  [23, 18, 13, 8, 3],
  [24, 19, 14, 9, 4],
  [25, 20, 15, 10, 5],
]
assert expected == rotate(g)

In [ ]:
# @title ##### 1.8 Given an NxN matrix, write a program to set the entire row and column to 0 if an element has a value of 0.
def zero_out(g):
  m, n = len(g), len(g[0])
  rows, columns = set(), set()
  for r in range(m):
    for c in range(n):
      if g[r][c] == 0:
        rows.add(r)
        columns.add(c)
  for r in range(m):
    for c in range(n):
      if r in rows or c in columns:
        g[r][c] = 0
  return g

g = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 0, 1, 2], [3, 4, 5, 6]]
assert [[1, 0, 3, 4], [5, 0, 7, 8], [0, 0, 0, 0], [3, 0, 5, 6]] == zero_out(g)

In [ ]:
# @title ##### 1.9 Given two strings, write a program to test if a string is a rotation of the other using isSubstring method.
is_rotated = lambda s, t: len(s) == len(t) and (s + s).find(t) > -1
assert all([is_rotated("x", "x"), is_rotated("xy", "yx"), is_rotated("xyz", "yzx")])
assert not is_rotated("xyz", "xyx")

In [ ]:
# @title ##### 2.1 Write a program to remove duplicates from an unordered linked list. What if you cannot use additional data structures?
from collections import namedtuple
from dataclasses import dataclass

@dataclass
class SNode:
  value: int = None
  next: "SNode" = None

  def __iter__(self):
    L = self
    while L:
      yield L
      L = L.next

  @classmethod
  def from_values(cls, *values):
    L = None
    for value in reversed(values):
      L = cls(value, L)
    return L

def dedup_o_n_time(L):
  curr, seen = L, {}
  while curr:
    if curr.value in seen:
      pred.next = curr.next
    else:
      seen[curr.value], pred = True, curr
    curr = curr.next
  return L

def dedup_o_1_space(L):
  lhs = L
  while lhs:
    pred = lhs  # predecessor of RHS.
    while pred.next:
      if lhs.value == pred.next.value:
        pred.next = pred.next.next
      else:
        pred = pred.next
    lhs = lhs.next
  return L

L3 = SNode.from_values(1, 2, 3)
assert L3 == dedup_o_n_time(SNode.from_values(1, 1, 2, 3, 3))
assert L3 == dedup_o_n_time(SNode.from_values(1, 2, 1, 2, 3))
assert L3 == dedup_o_1_space(SNode.from_values(1, 1, 2, 3, 3))
assert L3 == dedup_o_1_space(SNode.from_values(1, 2, 1, 2, 3))

In [ ]:
# @title ##### 2.2 Implement an algorithm to find the k-th to last element of a singly linked list.
def last(L : SNode, k=0):
  p, pk = L, None  # pk points to the k-th last node when p reaches the end.
  for _ in range(k):
    if p is None:
      break
    p = p.next
  if p:
    pk = L
    while p.next:
      p, pk = p.next, pk.next
  return pk


L4 = SNode.from_values(*range(4))  # 0, 1, 2, 3
assert [3, 2, 0, None] == [(_ := last(L4, k)) and _.value for k in (0, 1, 3, 4)]

In [ ]:
# @title ##### 2.3 Given access only to a node, implement an algorithm to delete that node in the middle of a singly linked list.

In [ ]:
# @title ##### 2.4 Write a program to partition a linked list around a value of x, such that all nodes less than x come before all nodes greater than or equal to x.
def partition(L, x):
  def push(pool, curr):
    curr.next = pool
    return curr
  def last(curr):
    while curr.next:
      curr = curr.next
    return curr
  lt_x, eq_x, gt_x, = [None], [None], [None]  # 1-element containers.
  curr = L
  while curr:
    next_ = curr.next
    pool = lt_x if curr.value < x else gt_x if curr.value > x else eq_x
    pool[0] = push(pool[0], curr)
    curr = next_
  last(lt_x[0]).next, last(eq_x[0]).next = eq_x[0], gt_x[0]
  return lt_x[0]

L9 = SNode.from_values(9, 3, 8, 2, 5, 6, 1, 7, 4, 5)
assert [4, 1, 2, 3, 5, 5, 7, 6, 8, 9] == [e.value for e in partition(L9, 5)]

2.5. Given two single linked lists where each node has a single digit, write a program that sums them up.

2.6 Write a program to test if a linked list is palindromic.

2.7 Given two linked lists that interacts by reference (not value), write a program to return the intersecting node.

2.8 Given a linked list, implement an algorithm which returns the node at the beginning of the loop. e.g., INPUT: a -> b -> c -> d -> e -> c, and OUTPUT: c.



3.1 How would you design and implement three stacks on a single array.

In [ ]:
# @title ###### 3.2 Design a stack that has **min** method that returns the minimum in addition to push and pop methods. Push, pop, and min should all operate in O(1) time.
class MinStack:
  def __init__(self):
    self.minimum, self.stack = None, []
  def push(self, e):
    if self.minimum is None or e <= self.minimum:
      self.stack.append(self.minimum)  # saves the previous minimum.
      self.minimum = e
    self.stack.append(e)
    return self
  def pop(self):
    if (e := self.stack.pop()) == self.minimum:
      self.minimum = self.stack.pop()
    return e

S = MinStack().push(2).push(3).push(2).push(1)
assert 1 == S.minimum and 1 == S.pop()
assert 2 == S.minimum and 2 == S.pop()
assert 2 == S.minimum and 3 == S.pop()
assert 2 == S.minimum and 2 == S.pop()
assert S.minimum is None

3.3 Imagine a literal stack of plates. If the stack gets too high, it might topple. Therefore, in real life, we would likely start a new stack when the previous stack exceeds some threshold. Implement a data structure SetOfStacks that mimics this. SetOfStacks should be composed of several stacks and should create a new stack once the previous one exceeds capacity. SetOfStacks.push() and SetOfStacks.pop() should behave identically to a single stack (that is, pop() should return the same values as it would if there were just a single stack). Implement a function popAt(int index) which performs a pop operation on a specific sub-stack.

In [ ]:
# @title ##### 3.5 Design a queue using two stacks.
class Q:
  def __init__(self, inbox=[], outbox=[]):
    self.inbox, self.outbox = inbox, outbox
  def offer(self, E):  # E: element.
    self.inbox.append(E)
    return self
  def poll(self):
    if not self.outbox:
      while self.inbox:
        self.outbox.append(self.inbox.pop())
    return self.outbox.pop() if self.outbox else None
  def __repr__(self):
    return f"Q({self.inbox!r}, {self.outbox!r})"

(q := Q()).offer(1).offer(2)
assert 1 == q.poll() and q.offer(3) is q
assert (2, 3, None) == tuple(q.poll() for _ in range(3))

In [ ]:
# @title ##### 3.6 Write a program to sort a stack so the largest elements are at the top. You may use additional stacks to hold items, but you may not copy the elements into any other data structure such as an array. The stack supports the following operations: push, pop, peek, and isEmpty.

def sort(S):  # a = [1, 2, 9, 8, 3, 4]
  L = []
  while S:
    e = S.pop()
    while L and L[-1] > e:
      S.append(L.pop())
    L.append(e)
  S.extend(L)
  return S

assert [1, 2, 3, 4, 5] == sort([1, 2, 5, 3, 4])

In [ ]:
def sort(S):
  L = []
  while S:
    e = S.pop()
    while L and L[-1] > e:
      S.append(L.pop())
    L.append(e)
  return L

3.7 An animal shelter holds only dogs and cats, and operations on a strictly "first in, first out" basis. People must adopt either the oldest (based on the arrival time) of all animals at the shelter, or they can select whether they would prefer a dog or a cat (and will receive the oldest animal of that type). They cannot select which speicific animal they would like. Create the data structures to maintain this system and implement operations such as enqueue, dequeueAny, dequeueDog, and dequeueCat. You may use the LinkedList data structure.